In [2]:
import torch 
import torch.nn.functional as F
from matplotlib import pyplot as plt 
%matplotlib inline

In [3]:
# Read the words
words = open('names.txt', 'r').read().splitlines()

# words[:8]
len(words)

32033

In [4]:
# Building the vocab of chars and mappings to and from integers
# Char to int
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0

# Int to Char
itos = {i:s for s,i in stoi.items()}

print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# Building the dataset
block_size = 3  # context length: how many characters do we take to predict the next one?
X, Y = [], []  # X = inputs and Y = Next word

for w in words[:5]:
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]   # Converted the chars to integers
        X.append(context)
        Y.append(ix)
        # print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]  # Crop and append (update the context window)

# Converting to tensors
X = torch.tensor(X)   
Y = torch.tensor(Y)

In [6]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [7]:
# Converting into embeddings
C = torch.randn((27, 2))  # Create a 27 x 2 tensor of randome numbers (embeddings)
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

### Implementing the hidden layer + internals of torch.Tensor: storage, views

In [8]:
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape  # cat(tensors, dimension)
# torch.cat(torch.unbind(emb, 1), 1).shape
# emb.view(32, 6).shape

In [ ]:
# w1 = torch.randn((6, 100))
# b1 = torch.randn(100)

In [10]:
h = emb.view(-1, 6) @ w1 + b1  # Hidden layer
h.shape   # SO here (I X Y) I = training examples, Y = hidden neurons

torch.Size([32, 100])

### Implementing the output layer

In [11]:
# Weights and biases for output layer
w2 = torch.randn((100, 27))  # Because there is 27 chars (., a-z)
b2 = torch.randn(27)

In [ ]:
# logits = h @ w2 + b2  # Output Lyaer
# logits.shape
# For each example (emm) network produces 27 numbers with some values and these numbers are called logits

torch.Size([32, 27])

### Implementing the negative log likelihood loss

In [13]:
counts = logits.exp()  # To convert -ve numbers to +ve
prob = counts / counts.sum(1, keepdims=True)  # Converting these +ve numbers into probabilities
prob.shape

torch.Size([32, 27])

In [ ]:
# loss = -prob[torch.arange(32), Y].log().mean()
# loss

tensor(inf)

### introducing F.cross_entropy

In [16]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [19]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)

W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)

W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]

In [20]:
sum(p.nelement() for p in parameters)  # numbers of parameters in total

3481

In [ ]:
emb = C[X]  
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2 

loss = F.cross_entropy(logits, Y)  
# loss
''' Better than performing whole expression because :
    forward and backward passes are much more efficient. 
'''

tensor(17.7697)

### Implementing the training loop, overfitting one batch

In [30]:
for p in parameters: 
    p.requires_grad = True 

In [36]:
for _ in range(100):

    # forward pass 
    emb = C[X]   # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
    logits = h @ W2 + b2   # (32, 27)
    loss = F.cross_entropy(logits, Y)

    # backward pass 
    for p in parameters:
        p.grad = None 
    loss.backward()

    # update 
    for p in parameters: 
        p.data += -0.1 * p.grad

print(loss.item())

0.25148823857307434
